In [1]:
# PDF Q&A System — Demo Notebook

In [3]:
# Import all required modules
from file_indexer import scan_folders, search_pdf_by_name, save_index
from pdf_processor import extract_pdf_pages
from vector_db_manager import create_chunks, create_vector_store, search_similar_chunks
from ollama_connector import get_answer
from folder_suggester import build_folder_embeddings, suggest_folders, check_answer_confidence
from colpali_processor import is_scanned_pdf

print("All modules imported successfully.")

Loading embedding model...


e:\Python\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\enggm\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2575.48it/s]


Model loaded! 
Loading embedding model for folder suggester...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4405.78it/s]


Folder suggester ready.
All modules imported successfully.


In [4]:
# Scan root folder for all PDFs
root_path = "./sample_pdfs"
pdf_index = scan_folders(root_path)
save_index(pdf_index)

print(f"Total PDFs found: {len(pdf_index)}")
print("\nPDF List:")
for pdf in pdf_index:
    print(f"  - {pdf['name']} → {pdf['folder']}")

Index saved! Total PDFs found: 4
Total PDFs found: 4

PDF List:
  - cpp_programming_tutorial.pdf → ./sample_pdfs\programming
  - java__programming_test.pdf → ./sample_pdfs\programming
  - python_programming_test.pdf → ./sample_pdfs\programming
  - PublicWaterMassMailing.pdf → ./sample_pdfs\scanned


In [5]:
# Search PDFs by keyword
search_query = "python"
results = search_pdf_by_name(pdf_index, search_query)

print(f"Search results for '{search_query}':")
for i, pdf in enumerate(results):
    print(f"  {i+1}. {pdf['name']}")

Search results for 'python':
  1. python_programming_test.pdf


In [6]:
# Select and process first result
selected_pdf = results[0]
print(f"Processing: {selected_pdf['name']}")

pages = extract_pdf_pages(selected_pdf["path"])
chunks = create_chunks(pages)

for chunk in chunks:
    chunk["pdf_name"] = selected_pdf["name"]

print(f"Total pages: {len(pages)}")
print(f"Total chunks: {len(chunks)}")

Processing: python_programming_test.pdf
Text-based PDF detected: ./sample_pdfs\programming\python_programming_test.pdf
  Page 1 processed.
  Page 2 processed.
  Page 3 processed.
  Page 4 processed.
  Page 5 processed.
  Page 6 processed.
  Page 7 processed.
  Page 8 processed.
  Page 9 processed.
  Page 10 processed.
  Page 11 processed.
  Page 12 processed.
  Page 13 processed.
  Page 14 processed.
  Page 15 processed.
  Page 16 processed.
  Page 17 processed.
  Page 18 processed.
  Page 19 processed.
  Page 20 processed.
  Page 21 processed.
  Page 22 processed.
  Page 23 processed.
  Page 24 processed.
  Page 25 processed.
  Page 26 processed.
  Page 27 processed.
  Page 28 processed.
  Page 29 processed.
  Page 30 processed.
  Page 31 processed.
  Page 32 processed.
  Page 33 processed.
  Page 34 processed.
  Page 35 processed.
  Page 36 processed.
  Page 37 processed.
  Page 38 processed.
  Page 39 processed.
  Page 40 processed.
  Page 41 processed.
  Page 42 processed.
  Page 4

In [7]:
# Create FAISS vector store
index, all_chunks = create_vector_store(chunks)
print(f"Vector store created successfully.")
print(f"Total vectors: {index.ntotal}")

Creating embeddings...


Batches: 100%|██████████| 5/5 [00:13<00:00,  2.71s/it]

vectors store created! Total vectors :147
Vector store created successfully.
Total vectors: 147


In [9]:
# Ask a question
query = "what is a function in python?"
print(f"Question: {query}")

relevant_chunks = search_similar_chunks(query, index, all_chunks)
answer = get_answer(query, relevant_chunks)

print(f"\nAnswer:\n{answer}")
print("\nAnswer retrieved from:")
for chunk in relevant_chunks:
    print(f"  - {chunk['pdf_name']} (Page {chunk['page_number']})")

Question: what is a function in python?
Generating answer using Ollama...

Answer:
A function in Python is a block of code which only runs when it is called. You can pass data (parameters) into a function and it can return data as a result. It has its own workspace and internal variables that are only valid inside the function.

Answer retrieved from:
  - python_programming_test.pdf (Page 64)
  - python_programming_test.pdf (Page 65)
  - python_programming_test.pdf (Page 2)
  - python_programming_test.pdf (Page 143)
  - python_programming_test.pdf (Page 40)


In [10]:
# Detect scanned vs text-based PDFs
print("PDF Type Detection:")
print("-" * 40)
for pdf in pdf_index:
    scanned = is_scanned_pdf(pdf["path"])
    status = "Scanned PDF" if scanned else "Text-based PDF"
    print(f"  {pdf['name']} → {status}")

PDF Type Detection:
----------------------------------------
Text-based PDF detected: ./sample_pdfs\programming\cpp_programming_tutorial.pdf
  cpp_programming_tutorial.pdf → Text-based PDF
Text-based PDF detected: ./sample_pdfs\programming\java__programming_test.pdf
  java__programming_test.pdf → Text-based PDF
Text-based PDF detected: ./sample_pdfs\programming\python_programming_test.pdf
  python_programming_test.pdf → Text-based PDF
Scanned PDF detected: ./sample_pdfs\scanned\PublicWaterMassMailing.pdf
  PublicWaterMassMailing.pdf → Scanned PDF


In [11]:
# Build folder embeddings
folder_data = build_folder_embeddings(root_path)

# Test folder suggestion
query = "what is water treatment?"
print(f"Query: {query}")

suggestions = suggest_folders(query, folder_data, current_folders=[])

if suggestions:
    print(f"\nSuggested folders:")
    for s in suggestions:
        print(f"  - {s['folder']} (Similarity: {s['similarity']:.0%})")
        for pdf in s["pdf_names"]:
            print(f"    → {pdf}")
else:
    print("No suggestions found.")

Building folder embeddings...
  Folder indexed: ./sample_pdfs\programming (3 PDFs)
  Folder indexed: ./sample_pdfs\scanned (1 PDFs)
Folder embeddings saved. Total folders: 2
Query: what is water treatment?

Suggested folders:
  - ./sample_pdfs\scanned (Similarity: 41%)
    → PublicWaterMassMailing


In [12]:
# Voice input demo
from voice_input import get_voice_query

print("Speak your question (10 seconds)...")
voice_query = get_voice_query(duration=10)
print(f"\nTranscribed: {voice_query}")

relevant_chunks = search_similar_chunks(voice_query, index, all_chunks)
answer = get_answer(voice_query, relevant_chunks)

print(f"\nAnswer:\n{answer}")
print("\nAnswer retrieved from:")
for chunk in relevant_chunks:
    print(f"  - {chunk['pdf_name']} (Page {chunk['page_number']})")

Loading Whisper model...
Whisper model loaded successfully.
Speak your question (10 seconds)...

Recording for 10 seconds...
Speak now...
Recording completed.
Converting speech to text...


e:\Python\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


You said: 

Transcribed: 
Generating answer using Ollama...

Answer:
Answer not found in document.

Answer retrieved from:
  - python_programming_test.pdf (Page 9)
  - python_programming_test.pdf (Page 20)
  - python_programming_test.pdf (Page 96)
  - python_programming_test.pdf (Page 2)
  - python_programming_test.pdf (Page 143)


In [ ]:
# Scanned PDF with Auto Processor

from scanned_handler import is_scanned_pdf, extract_text_from_scanned_pdf
import torch

print(f"GPU Available: {torch.cuda.is_available()}")
print(f"Processor: {'ColPali' if torch.cuda.is_available() else 'Tesseract OCR'}")
print("-" * 40)

# Test scanned PDF
scanned_pdf = "./sample_pdfs/scanned/PublicWaterMassMailing.pdf"
scanned = is_scanned_pdf(scanned_pdf)
print(f"Is scanned: {scanned}")

if scanned:
    pages = extract_text_from_scanned_pdf(scanned_pdf)
    print(f"Total pages processed: {len(pages)}")
    print(f"\nFirst page preview:")
    print(pages[0]["text_content"][:200])